In [1]:
import pandas as pd
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.schema.output_parser import StrOutputParser
from langchain.chat_models import init_chat_model


In [2]:

llm = init_chat_model("gpt-4.1")


In [3]:
df = pd.read_excel(r"C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\Context-recall-eval-dataset.xlsx")
questions = df['user_input'].tolist()
ground_truth_lookup = dict(zip(df['user_input'], df['reference']))



In [4]:


template = ChatPromptTemplate.from_messages([
    ("system", "You are an intelligent assistant who answers questions about a given document."),
    ("user", "Here is the document content: {context}. My question is: {question}")
])
chain = template | llm | StrOutputParser()

In [5]:
import os
import json
from langchain_community.docstore.document import Document

DOCS_CACHE_FILE = r"C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\all_collected_docs_cache_new.json"

if os.path.exists(DOCS_CACHE_FILE): # Start of cache check block
        print(f"Loading documents from cache: {DOCS_CACHE_FILE}")
        with open(DOCS_CACHE_FILE, 'r', encoding='utf-8') as f:
            cached_data = json.load(f)
            all_collected_docs = [
                Document(page_content=item['page_content'], metadata=item['metadata'])
                for item in cached_data
            ]

Loading documents from cache: C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\all_collected_docs_cache_new.json


In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:

from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

RETRIEVAL_RESULTS_CACHE_FILE = r"C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\retrieval_results_2.json"
os.makedirs(os.path.dirname(RETRIEVAL_RESULTS_CACHE_FILE), exist_ok=True)



chunk_sizes = [1000,1200]
chunk_overlaps = [200,250]
all_chunks_keys_list = [] # This will store names like "Recursive_256_0"
for cs in chunk_sizes:
    for co in chunk_overlaps:
        all_chunks_keys_list.append(f"Recursive_{cs}_{co}")

retrieval_results = {}
    # The main loop for evaluating each chunking strategy
for current_chunk_size in chunk_sizes:
    for current_chunk_overlap in chunk_overlaps:
        # --- RecursiveCharacterTextSplitter ---
        config_name_recursive = f"Recursive_{current_chunk_size}_{current_chunk_overlap}"
        search_index_recursive = f"new-test-rag-eval-recursive-{current_chunk_size}-{current_chunk_overlap}".replace("_", "-") # Unique index name

        if config_name_recursive not in retrieval_results:
            print(f"\n--- Processing new config: {config_name_recursive} (Index: {search_index_recursive}) ---")

            # 1. Chunk documents based on current config
            splitter_recursive = RecursiveCharacterTextSplitter(
                chunk_size=current_chunk_size,
                chunk_overlap=current_chunk_overlap,
                length_function=len,
                is_separator_regex=False,
            )
            # Use original `all_collected_docs` for re-chunking
            current_chunks_for_index = splitter_recursive.split_documents(all_collected_docs)
            print(f"Split source documents into {len(current_chunks_for_index)} chunks for {config_name_recursive}.")

            try:
                print(f"Creating/Populating index: {search_index_recursive} with {len(current_chunks_for_index)} documents...")


                from langchain_classic.vectorstores import Chroma

                persist_path = f"./chroma_store/{config_name_recursive}"

                if os.path.exists(persist_path):
                    #  Load existing vector store
                    chroma_vectordb = Chroma(
                        persist_directory=persist_path,
                        embedding_function=embeddings
                    )
                else:
                    #  Create new vector store from documents
                    chroma_vectordb = Chroma.from_documents(
                        documents=current_chunks_for_index,
                        embedding=embeddings,
                        persist_directory=persist_path
                    )
                    chroma_vectordb.persist()

                chroma_retriever = chroma_vectordb.as_retriever(
                    search_kwargs={"k": 2}
)

                print(f"Index {search_index_recursive} populated.")
                config_results = []
                from tqdm import tqdm
                # for question in questions: # Iterate through your test questions
                for question in tqdm(
    questions,
    desc=f"Questions ({config_name_recursive})",
    leave=False
):

                    try:
                        retrieved_docs = chroma_retriever.invoke(question)
                        doc_contents = [doc.page_content for doc in retrieved_docs]
                        print(f"/n Retrieved {len(doc_contents)} documents for question: '{question}' using {config_name_recursive}/n")
                        # print(f"Document contents: {doc_contents}")
                        combined_context = " ".join(doc_contents) if doc_contents else "No relevant context found."

                        result_llm_response = chain.invoke({"context": combined_context, "question": question})
                        answer_str = result_llm_response.content if hasattr(result_llm_response, 'content') else str(result_llm_response)

                        config_results.append({
                            "question": question,
                            "answer": answer_str,
                            "contexts": doc_contents
                        })
                    except Exception as e:
                        print(f"Error processing question '{question}' for {config_name_recursive}: {e}")
                        config_results.append({
                            "question": question,
                            "answer": "Error in retrieval",
                            "contexts": []
                        })
                retrieval_results[config_name_recursive] = config_results

            except Exception as e:
                print(f"ERROR: Could not initialize/populate Azure Search for {config_name_recursive}: {e}")
                # You might want to add empty results or mark this config as failed
                retrieval_results[config_name_recursive] = [] # Mark as empty if error
        else:
            print(f"Skipping config: {config_name_recursive} (already in cache)")


--- Processing new config: Recursive_1000_200 (Index: new-test-rag-eval-recursive-1000-200) ---
Split source documents into 223 chunks for Recursive_1000_200.
Creating/Populating index: new-test-rag-eval-recursive-1000-200 with 223 documents...


C:\Users\abhim\AppData\Local\Temp\ipykernel_16888\374001544.py:47: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_vectordb = Chroma(


Index new-test-rag-eval-recursive-1000-200 populated.


Questions (Recursive_1000_200):   0%|          | 0/6 [00:00<?, ?it/s]

/n Retrieved 2 documents for question: 'What Seventh Edition about?' using Recursive_1000_200/n


...

/n Retrieved 2 documents for question: 'How does Bayes' Theorem explain the low probab...' using Recursive_1200_250/n


Questions (Recursive_1200_250):  67%|██████▋   | 4/6 [00:31<00:18,  9.50s/it]

/n Retrieved 2 documents for question: 'How does Bayes’ Theorem facilitate the reversa...' using Recursive_1200_250/n


Questions (Recursive_1200_250):  83%|████████▎ | 5/6 [00:38<00:08,  8.84s/it]

/n Retrieved 2 documents for question: 'How does the volatility of the NASDAQ index in...' using Recursive_1200_250/n


In [12]:
with open(RETRIEVAL_RESULTS_CACHE_FILE, "w", encoding='utf-8') as f:
        json.dump(retrieval_results, f, indent=2, ensure_ascii=False)
print(f"Final retrieval results cached to {RETRIEVAL_RESULTS_CACHE_FILE}")

Final retrieval results cached to C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\retrieval_results_2.json


In [15]:
from ragas.metrics import context_recall,context_precision,faithfulness


C:\Users\abhim\AppData\Local\Temp\ipykernel_16888\3286909078.py:1: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall,context_precision,faithfulness
C:\Users\abhim\AppData\Local\Temp\ipykernel_16888\3286909078.py:1: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_recall,context_precision,faithfulness
C:\Users\abhim\AppData\Local\Temp\ipykernel_16888\3286909078.py:1: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faith

In [17]:
from datasets import Dataset
from ragas.metrics import context_recall,context_precision,faithfulness
from ragas import evaluate
# print(f"\nLoading retrieval results from cache: {RETRIEVAL_RESULTS_CACHE_FILE}")
with open(RETRIEVAL_RESULTS_CACHE_FILE, 'r', encoding='utf-8') as f:
    retrieval_results = json.load(f)


context_recall_result = {}
for config_key in all_chunks_keys_list:
    eval_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}
    if config_key in retrieval_results:
        for item in retrieval_results[config_key]:
            eval_data["question"].append(item["question"])
            eval_data["answer"].append(item["answer"])
            eval_data["contexts"].append(item["contexts"])
            gt = ground_truth_lookup.get(item["question"], "")
            eval_data["ground_truth"].append(" ".join(gt) if isinstance(gt, list) else str(gt or ""))
    else:
        print(f"Warning: No retrieval results found for config '{config_key}'. Skipping for Ragas dataset preparation.")

    if not eval_data["question"]:
        print("Error: No data prepared for Ragas evaluation. Exiting.")
        exit(1)

    print(f"\nEvaluation dataset size for Ragas: {len(eval_data['question'])}")
    for i in range(min(5, len(eval_data["question"]))):
        print(f"Sample {i}: Question='{eval_data['question'][i]}', Answer='{eval_data['answer'][i][:50]}...', Contexts Count={len(eval_data['contexts'][i])}, Ground Truth='{eval_data['ground_truth'][i]}'")

    ragas_dataset_for_eval = Dataset.from_dict(eval_data)

    # --- Evaluate using RAGAS ---
    print("\n--- Starting Ragas Evaluation ---")
    try:
        ragas_evaluation_result = evaluate(
            dataset=ragas_dataset_for_eval,
            metrics=[context_recall,context_precision,faithfulness],
            llm=llm,
            raise_exceptions=True
        )
        print("ragas_evaluation_result:/n",ragas_evaluation_result)
    except Exception as e:
        print(f"Error during RAGAS evaluation: {e}")
        exit(1)
    print("\nRAGAS Evaluation Complete.")

    # --- Store Aggregate Scores ---
    # aggregate_scores = ragas_evaluation_result
    context_recall_result[config_key] = ragas_evaluation_result
    # context_recall_result[config_key] = dict(ragas_evaluation_result.scores) # Store the dict of aggregate scores
    print("context_recall_result: /n",context_recall_result)
    df = pd.DataFrame.from_dict(context_recall_result, orient='index')
    print(df.head())
    print("/n------------------------------------------------------/n")
    # break

C:\Users\abhim\AppData\Local\Temp\ipykernel_16888\268441452.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall,context_precision,faithfulness
C:\Users\abhim\AppData\Local\Temp\ipykernel_16888\268441452.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_recall,context_precision,faithfulness
C:\Users\abhim\AppData\Local\Temp\ipykernel_16888\268441452.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithful


Evaluation dataset size for Ragas: 6
Sample 0: Question='What Seventh Edition about?', Answer='The Seventh Edition referred to in your excerpt is...', Contexts Count=2, Ground Truth='The Seventh Edition refers to 'Complete Busine...'
Sample 1: Question='What was the stock price percentage for Lehman...', Answer='According to the provided document, the stock pric...', Contexts Count=2, Ground Truth='The stock price percentage for Lehman Brothers...'
Sample 2: Question='How do business statistics relate to calculati...', Answer='Business statistics play a crucial role in calcula...', Contexts Count=2, Ground Truth='Business statistics involve the use of mathema...'
Sample 3: Question='How does Bayes' Theorem explain the low probab...', Answer='Your question was cut off, but it appears you are ...', Contexts Count=2, Ground Truth='Bayes' Theorem explains the low probability of...'
Sample 4: Question='How does Bayes’ Theorem facilitate the reversa...', Answer='Bayes’ Theorem facilitates t

Evaluating: 100%|██████████| 18/18 [01:20<00:00,  4.47s/it]


ragas_evaluation_result:/n {'context_recall': 0.6250, 'context_precision': 1.0000, 'faithfulness': 0.6844}

RAGAS Evaluation Complete.
context_recall_result: /n {'Recursive_1000_200': {'context_recall': 0.6250, 'context_precision': 1.0000, 'faithfulness': 0.6844}}
                                                               scores  \
Recursive_1000_200  [{'context_recall': 1.0, 'context_precision': ...   

                                                              dataset  \
Recursive_1000_200  {'samples': [user_input='What Seventh Edition ...   

                   binary_columns cost_cb  \
Recursive_1000_200             []    None   

                                                               traces  \
Recursive_1000_200  [{'scores': {'context_recall': 1.0, 'context_p...   

                                                         ragas_traces run_id  
Recursive_1000_200  {'019c438e-9745-75f0-8a36-394efdb0dfa5': run_i...   None  
/n-------------------------------------------

Evaluating: 100%|██████████| 18/18 [01:27<00:00,  4.86s/it]


ragas_evaluation_result:/n {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.6550}

RAGAS Evaluation Complete.
context_recall_result: /n {'Recursive_1000_200': {'context_recall': 0.6250, 'context_precision': 1.0000, 'faithfulness': 0.6844}, 'Recursive_1000_250': {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.6550}}
                                                               scores  \
Recursive_1000_200  [{'context_recall': 1.0, 'context_precision': ...   
Recursive_1000_250  [{'context_recall': 1.0, 'context_precision': ...   

                                                              dataset  \
Recursive_1000_200  {'samples': [user_input='What Seventh Edition ...   
Recursive_1000_250  {'samples': [user_input='What Seventh Edition ...   

                   binary_columns cost_cb  \
Recursive_1000_200             []    None   
Recursive_1000_250             []    None   

                                                        

Evaluating: 100%|██████████| 18/18 [01:29<00:00,  4.99s/it]


ragas_evaluation_result:/n {'context_recall': 0.6333, 'context_precision': 1.0000, 'faithfulness': 0.5717}

RAGAS Evaluation Complete.
context_recall_result: /n {'Recursive_1000_200': {'context_recall': 0.6250, 'context_precision': 1.0000, 'faithfulness': 0.6844}, 'Recursive_1000_250': {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.6550}, 'Recursive_1200_200': {'context_recall': 0.6333, 'context_precision': 1.0000, 'faithfulness': 0.5717}}
                                                               scores  \
Recursive_1000_200  [{'context_recall': 1.0, 'context_precision': ...   
Recursive_1000_250  [{'context_recall': 1.0, 'context_precision': ...   
Recursive_1200_200  [{'context_recall': 1.0, 'context_precision': ...   

                                                              dataset  \
Recursive_1000_200  {'samples': [user_input='What Seventh Edition ...   
Recursive_1000_250  {'samples': [user_input='What Seventh Edition ...   
Recursive_1200_20

Evaluating: 100%|██████████| 18/18 [01:14<00:00,  4.12s/it]


ragas_evaluation_result:/n {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.7160}

RAGAS Evaluation Complete.
context_recall_result: /n {'Recursive_1000_200': {'context_recall': 0.6250, 'context_precision': 1.0000, 'faithfulness': 0.6844}, 'Recursive_1000_250': {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.6550}, 'Recursive_1200_200': {'context_recall': 0.6333, 'context_precision': 1.0000, 'faithfulness': 0.5717}, 'Recursive_1200_250': {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.7160}}
                                                               scores  \
Recursive_1000_200  [{'context_recall': 1.0, 'context_precision': ...   
Recursive_1000_250  [{'context_recall': 1.0, 'context_precision': ...   
Recursive_1200_200  [{'context_recall': 1.0, 'context_precision': ...   
Recursive_1200_250  [{'context_recall': 1.0, 'context_precision': ...   

                                                            

In [18]:
print(context_recall_result)

{'Recursive_1000_200': {'context_recall': 0.6250, 'context_precision': 1.0000, 'faithfulness': 0.6844}, 'Recursive_1000_250': {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.6550}, 'Recursive_1200_200': {'context_recall': 0.6333, 'context_precision': 1.0000, 'faithfulness': 0.5717}, 'Recursive_1200_250': {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.7160}}


In [22]:
df = pd.DataFrame.from_dict(context_recall_result,orient="index")
df

,scores,dataset,binary_columns,cost_cb,traces,ragas_traces,run_id
Recursive_1000_200,"[{'context_recall': 1.0, 'context_precision': ...",{'samples': [user_input='What Seventh Edition ...,[],None,"[{'scores': {'context_recall': 1.0, 'context_p...",{'019c438e-9745-75f0-8a36-394efdb0dfa5': run_i...,None
Recursive_1000_250,"[{'context_recall': 1.0, 'context_precision': ...",{'samples': [user_input='What Seventh Edition ...,[],None,"[{'scores': {'context_recall': 1.0, 'context_p...",{'019c438f-dd32-7650-a795-2bd800e9410d': run_i...,None
Recursive_1200_200,"[{'context_recall': 1.0, 'context_precision': ...",{'samples': [user_input='What Seventh Edition ...,[],None,"[{'scores': {'context_recall': 1.0, 'context_p...",{'019c4391-3ea0-7a61-8494-5e47e6137e3d': run_i...,None
Recursive_1200_250,"[{'context_recall': 1.0, 'context_precision': ...",{'samples': [user_input='What Seventh Edition ...,[],None,"[{'scores': {'context_recall': 1.0, 'context_p...",{'019c4392-a923-76d2-9ed6-def856072bed': run_i...,None


In [ ]:
df.to_csv(r"C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\ragas_evaluation_results.csv", index=True)

In [21]:
variable = {'Recursive_1000_200': {'context_recall': 0.6250, 'context_precision': 1.0000, 'faithfulness': 0.6844}, 'Recursive_1000_250': {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.6550}, 'Recursive_1200_200': {'context_recall': 0.6333, 'context_precision': 1.0000, 'faithfulness': 0.5717}, 'Recursive_1200_250': {'context_recall': 0.5917, 'context_precision': 1.0000, 'faithfulness': 0.7160}}
df_new = pd.DataFrame.from_dict(variable,orient="index")
df_new

,context_recall,context_precision,faithfulness
Recursive_1000_200,0.6250,1.0,0.6844
Recursive_1000_250,0.5917,1.0,0.6550
Recursive_1200_200,0.6333,1.0,0.5717
Recursive_1200_250,0.5917,1.0,0.7160


In [ ]:
df_new.to_csv(r"C:\Users\abhim\OneDrive\Documents\Projects\RAG evaluation with RAGAS\ragas_evaluation_results2.csv", index=True)